## Annotate variants with VEP

https://www.ensembl.org/info/docs/tools/vep/index.html


For single sample or small sample set experiment, you can use VEP web interface.

For large sample set or batch analysis, consider using the commandline version.

You can install VEP with conda

`conda install ensembl-vep`

Noted* The conda install is not maintain by Ensemble VEP team.

**Install human annotation database**

In [ ]:
vep_install -a cf \ # Download cach and FASTA
    -s homo_sapiens \
    -y GRCh38 \ # Specific assemble version
    -c /path/to/GRCh38/vep \
    --CONVERT

**Caution**

You need to have available storage ~50-100GB during install process.

The final databased may take ~26GB.

---
To run VEP

In [ ]:
vep -i input.vcf -o output_annotated.vcf \
    --cache \
    --dir_cache /path/to/your/cache_directory \
    --assembly GRCh38 \
    --vcf \
    --hgvs --af --af_gnomad --max_af \
    --force_overwrit

---

## Annotate SV with AnnotSV

https://lbgi.fr/AnnotSV/

You can install annotSV with conda

`conda install annotsv`

You also need to download annoation database separately.

The database require ~50GB disk space

Run the install script provide in AnnotSV git repository.

https://github.com/lgmgeo/AnnotSV/blob/master/bin/INSTALL_annotations.sh



INSTALL_annotations.sh

In [ ]:
#!/bin/bash

############################################################
# Installing AnnotSV human annotations in a local directory
############################################################

# USAGE:
########
# INSTALL_annotations.sh "Version of AnnotSV human annotation" "Version of Exomiser phenotype annotations"

# AIM:
######
# Download Exomiser and AnnotSV annotations to be used with the "-annotationsDir" option

# CONTEXT:
##########
# To work with bioconda/docker/singularity, AnnotSV couldn't contain the annotations in the recipe (that would make the recipe very large, which is a bad practice in bioconda)
# Users need to download the annotation files once and pass the directory to AnnotSV at runtime with the "-annotationsDir" option.


mkdir AnnotSV_annotations
cd AnnotSV_annotations

# AnnotSV annotations
echo ""
echo "Download AnnotSV supporting data files:"
echo ""
curl -C - -LO https://www.lbgi.fr/~geoffroy/Annotations/Annotations_Human_$1.tar.gz
tar -xf Annotations_Human_$1.tar.gz -C ./
rm -rf Annotations_Human_$1.tar.gz

# Exomiser
echo ""
echo "Download Exomiser supporting data files:"
echo ""
curl -C - -LO https://data.monarchinitiative.org/exomiser/data/$2_phenotype.zip
unzip $2_phenotype.zip -d Annotations_Exomiser/$2/
rm -rf 2406_phenotype.zip

chmod -R 777 ./Annotations_*

**Run the script in terminal**

You need to specify 2 argument to the script

1. Human annotation database version
2. Exomiser database version

You can check Human annoation database version from Makefile

https://github.com/lgmgeo/AnnotSV/blob/master/Makefile

Looking for `HUMANVERSION`
and
Looking for `install-exomiser-1`



In [ ]:
./INSTALL_annotations.sh 3.5 2406

To enable exomiser

If we install program with conda, additional config file is needed.

Download config file from AnnotSV git repo and place it to `/etc/AnnotSV/`.

Which locate inside your conda environment.

In [ ]:
curl -o /path/to/miniconda3/envs/${Your_env_name}/etc/AnnotSV/application.properties \
    https://raw.githubusercontent.com/lgmgeo/AnnotSV/master/etc/AnnotSV/application.properties

Edit file `application.properties` by
 - Specify value of `exomiser.data-directory` to path that point to exomiser database.
 - Specify version of exomiser (eg. 2406)

Example

In [ ]:
#
# The Exomiser - A tool to annotate and prioritize genomic variants
#
# Copyright (c) 2016-2018 Queen Mary University of London.
# Copyright (c) 2012-2016 Charité Universitätsmedizin Berlin and Genome Research Ltd.
#
# This program is free software: you can redistribute it and/or modify
# it under the terms of the GNU Affero General Public License as
# published by the Free Software Foundation, either version 3 of the
# License, or (at your option) any later version.
#
# This program is distributed in the hope that it will be useful,
# but WITHOUT ANY WARRANTY; without even the implied warranty of
# MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
# GNU Affero General Public License for more details.
#
# You should have received a copy of the GNU Affero General Public License
# along with this program.  If not, see <http://www.gnu.org/licenses/>.
#
spring.application.name=exomiser-prioritiser-service

server.servlet.context-path=/exomiser/api/prioritise
server.port=XXXX
server.servlet.application-display-name=Exomiser Prioritiser Server

#Absolute system path where the exomiser data is installed
#exomiser.data-directory=${project.build.testOutputDirectory}
exomiser.data-directory=/Users/Jane/db/AnnotSV_annotations/Annotations_Exomiser
exomiser.phenotype.data-version=2406
exomiser.phenotype.random-walk-preload=true

#Actuator configuration
info.name=${server.display-name}
info.build.version=${project.version}
info.build.timestamp=${build.timestamp}
~                                      

---
To run annotSV

In [ ]:
AnnotSV -genomeBuild GRCh38 \
    -SVinputFile input_vcf \
    -outputFile output_tsv \
    -annotationDir /path/to/AnnotSV_annotations 

---
**Noted**

Both VEP and annotSV are not support .bcf format.

We have to convert BCF to VCF.

In [ ]:
bcftools view -o output_vcf input_vcf

---

## IGV session

Download data for this session

https://drive.google.com/file/d/1WNwvqPJLaz8nJDtgYYUXeOGuVY-nYdlv/view?usp=sharing